# 27 — Multi-Provider Fan-Out

Same question, three models, one answer. `ThreadPoolExecutor` fires all model calls in parallel via OpenRouter. Each opinion is typed. A synthesis pass produces a `ModelConsensus` with agreement, disagreement, and a consolidated recommendation.

In [ ]:
!pip install -q openai python-dotenv pydantic

In [ ]:
import os
os.environ['OPENROUTER_API_KEY'] = 'sk-or-...'  # replace

In [ ]:
from pydantic import BaseModel, Field

class StrategicOpinion(BaseModel):
    model: str = Field(description='Model that produced this opinion.')
    recommendation: str = Field(description='Top strategic recommendation (1-2 sentences).')
    key_risks: list[str] = Field(description='Up to 3 key risks.')
    key_opportunities: list[str] = Field(description='Up to 3 key opportunities.')
    confidence: str = Field(description='high, medium, or low.')

class ModelConsensus(BaseModel):
    topic: str = Field(description='Strategic topic.')
    opinions: list[StrategicOpinion] = Field(description='One opinion per model.')
    points_of_agreement: list[str] = Field(description='Where models agree.')
    points_of_disagreement: list[str] = Field(description='Where models diverge.')
    synthesised_recommendation: str = Field(description='Consolidated recommendation.')

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

MODELS = ['openai/gpt-4o-mini', 'mistralai/mistral-7b-instruct', 'meta-llama/llama-3.1-8b-instruct']
SYNTHESIS_MODEL = 'openai/gpt-4o-mini'

_OPINION_SYS = 'You are a strategic advisor. Give a recommendation (1-2 sentences), up to 3 risks, up to 3 opportunities, and your confidence level (high/medium/low). Be specific.'
_SYNTHESIS_SYS = 'Synthesise multiple strategic opinions. Identify agreement, disagreement, and produce one consolidated recommendation.'

def _query(client, model, topic):
    r = client.beta.chat.completions.parse(model=model, messages=[{'role':'system','content':_OPINION_SYS},{'role':'user','content':f'Topic: {topic}'}], response_format=StrategicOpinion)
    op = r.choices[0].message.parsed
    op.model = model
    return op

def run(topic):
    client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=os.environ['OPENROUTER_API_KEY'])
    opinions = []
    with ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
        futures = {ex.submit(_query, client, m, topic): m for m in MODELS}
        for f in as_completed(futures):
            opinions.append(f.result())
    opinions.sort(key=lambda o: MODELS.index(o.model))
    opinions_text = '\n\n'.join(f'Model: {o.model}\nRec: {o.recommendation}\nRisks: {o.key_risks}\nOps: {o.key_opportunities}\nConf: {o.confidence}' for o in opinions)
    r = client.beta.chat.completions.parse(model=SYNTHESIS_MODEL, messages=[{'role':'system','content':_SYNTHESIS_SYS},{'role':'user','content':f'Topic: {topic}\n\n{opinions_text}'}], response_format=ModelConsensus)
    consensus = r.choices[0].message.parsed
    consensus.topic = topic
    consensus.opinions = opinions
    return consensus

In [ ]:
topic = 'Should a mid-market B2B SaaS company prioritise EMEA expansion or double down on the US market?'
consensus = run(topic)
print(f'Topic: {consensus.topic}')
for o in consensus.opinions:
    print(f'\n[{o.model}] {o.confidence}')
    print(f'  {o.recommendation}')
if consensus.points_of_agreement:
    print('\nAgreement:')
    for p in consensus.points_of_agreement:
        print(f'  + {p}')
if consensus.points_of_disagreement:
    print('\nDisagreement:')
    for p in consensus.points_of_disagreement:
        print(f'  ! {p}')
print(f'\nConsolidated: {consensus.synthesised_recommendation}')

## Exercise

Swap one model string in `MODELS` for a different OpenRouter model (e.g. `anthropic/claude-haiku`). Re-run and see whether the consensus shifts. The schema and synthesis step are unchanged — only the model string changes.